In [1]:
import sys
sys.path.append('/home/fengw666/CT_Recon/dip-ct-benchmark')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import numpy as np
import argparse
import odl
from dival.data import DataPairs
from dival.evaluation import TaskTable
from dival.measure import PSNR, SSIM
from dival.reconstructors.odl_reconstructors import FBPReconstructor
from dival import get_standard_dataset
from dliplib.utils.helper import load_standard_dataset
from dliplib.utils.helper import set_use_latex
import torch
from dival.util.plot import plot_images
from hyreconstructor import HybirdReconstructor
from dliplib.reconstructors.tv import TVReconstructor
from dliplib.reconstructors.dip import DeepImagePriorReconstructor
from dival.reference_reconstructors import (
    check_for_params, download_params, get_params_path)
import matplotlib.pyplot as plt
from dliplib.utils import Params
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
import pydicom
from utils import normalize_
set_use_latex()

np.random.seed(9527)
torch.manual_seed(9527)
torch.cuda.manual_seed_all(9527)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True


In [2]:
def get_parser():
    """Adds arguments to the command"""
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', type=str, default='lodopab')
    parser.add_argument('--pid', type=str, default='1')
    parser.add_argument('--num_angles', type=int, default=1000)
    parser.add_argument('--impl', type=str, default='astra_cuda')
    parser.add_argument('--photons', type=int, default=1000)
    parser.add_argument('--layers', type=int, default=30)
    parser.add_argument('--iters', type=int, default=2000)
    return parser

def hy_callback_func(iteration, reconstruction, initial_reco, gt, loss, pnsr, ssim, prefix, save_path):
    _, ax = plot_images([initial_reco, reconstruction, gt], xticks=[], yticks=[], 
                        vrange='individual',
                        cbar='none', fig_size=(10, 4))
    ax[0].set_title('FBP')
    ax[1].set_xlabel('loss: {:f}; PNSR: {:2f} SSIM: {:f}'.format(loss, pnsr, ssim))
    ax[1].set_title('iteration {:d}'.format(iteration+1))
    ax[2].set_title('ground truth')
    plt.tight_layout()
    plt.savefig(save_path + 'best_result_{}.pdf'.format(prefix))
    plt.close()

In [3]:

# options = get_parser().parse_args()
TASK_NAME = 'lodopab'
IMPL = 'astra_cuda'
NUM_ANGLES  = 1000
PHOTOS_PER_PIXEL = 1000
LAYERS = 30
PID = '1'
ITERS = 2000
recons = {}
MU_AIR = 0.02
MU_WATER = 0.02
SAMPLE = 9

if TASK_NAME.lower() == 'ellipses':
    save_path = './layers{}/reco_{}_dose{}_ang{}/'.format(LAYERS, TASK_NAME, PHOTOS_PER_PIXEL, NUM_ANGLES)
    if not os.path.exists(save_path):
        os.mkdir(save_path)
    ellipses_dataset = load_standard_dataset('ellipses')
    reco_space = odl.uniform_discr(min_pt=[-90, -90], max_pt=[90, 90], shape=[360, 360], dtype='float32')
    ground_truth = odl.phantom.shepp_logan(reco_space, modified=True)
    ground_truth = np.asarray(ground_truth)
    ground_truth = normalize_(ground_truth, ground_truth.min(), ground_truth.max())
elif TASK_NAME.lower() == 'lodopab':
    save_path = './layers{}/reco_{}_dose{}_ang{}_sample{}_TimeCal/'.format(LAYERS, TASK_NAME, PHOTOS_PER_PIXEL, NUM_ANGLES, SAMPLE)
    if not os.path.exists(save_path):
        os.mkdir(save_path)
    dataset = get_standard_dataset('lodopab', impl=IMPL)
    _, ground_truth = dataset.get_sample(SAMPLE, 'test')
    ground_truth = np.asarray(ground_truth)
    ground_truth = normalize_(ground_truth, ground_truth.min(), ground_truth.max())
    reco_space = odl.uniform_discr(min_pt=[-90, -90], max_pt=[91, 91], shape=[362, 362], dtype='float32')
elif TASK_NAME.lower() == 'mayo':
    save_path = './layers{}/reco_{}_PID_{}_dose{}_iters{}/'.format(LAYERS, TASK_NAME, PID, PHOTOS_PER_PIXEL, ITERS)
    if not os.path.exists(save_path):
        os.mkdir(save_path)
    if PID == '1':
        imafile = '/home/fengw666/Low_Does_Dataset/L067/full_1mm/L067_FD_1_1.CT.0001.0001.2015.12.22.18.09.40.840353.358074219.IMA'
    elif PID == '2':
        imafile = '/home/fengw666/Low_Does_Dataset/L067/full_1mm/L067_FD_1_1.CT.0001.0306.2015.12.22.18.09.40.840353.358081539.IMA'
    elif PID == '3':
        imafile = '/home/fengw666/Low_Does_Dataset/L067/full_1mm/L067_FD_1_1.CT.0001.0559.2015.12.22.18.09.40.840353.358089994.IMA'
    else:
        raise NameError('Selection Is Incorrect')
    reco_space = odl.uniform_discr(min_pt=[-128, -128], max_pt=[128, 128], shape=[512, 512], dtype='float32')
    dataset = pydicom.read_file(imafile)
    img = dataset.pixel_array.astype(np.float32)
    # RescaleSlope = dataset.RescaleSlope
    # RescaleIntercept = dataset.RescaleIntercept
    # CT_img = img * RescaleSlope + RescaleIntercept
    ground_truth = normalize_(img, img.min(), img.max())
else:
    raise NameError('Task Name Is Incorrect')

angle_partition = odl.uniform_partition(0, 2 * np.pi, NUM_ANGLES)
detector_partition = odl.uniform_partition(-360, 360, 1000)
geometry = odl.tomo.FanBeamGeometry(angle_partition, detector_partition, src_radius=500, det_radius=500)
ray_trafo = odl.tomo.RayTransform(reco_space, geometry, impl=IMPL)

proj_data = ray_trafo(ground_truth)
proj_data = np.exp(-proj_data * MU_WATER)
proj_data = odl.phantom.poisson_noise(proj_data * PHOTOS_PER_PIXEL)
proj_data = np.maximum(proj_data, 1) / PHOTOS_PER_PIXEL
observation = np.log(proj_data) * (-1 / MU_WATER)

#####################Compared method########################################
test_data = DataPairs(observation, ground_truth, name='tv+dip')
# task table and reconstructors
eval_tt = TaskTable()



In [4]:
fbp_reconstructor = FBPReconstructor(ray_trafo, hyper_params={
                                        'filter_type': 'Hann',
                                        'frequency_scaling': 0.8})
if TASK_NAME.lower() == 'ellipses':
    params_tv = Params.load('ellipses_tv')
    params_dip = Params.load('ellipses_dip')
elif TASK_NAME.lower() == 'lodopab':
    params_tv = Params.load('lodopab_tv')
    params_dip = Params.load('lodopab_dip')
    params_tv.loss_function = 'mse'
    params_dip.loss_function = 'mse'
elif TASK_NAME.lower() == 'mayo':
    params_tv = Params.load('lodopab_tv')
    params_dip = Params.load('lodopab_dip')
    params_tv.loss_function = 'mse'
    params_dip.loss_function = 'mse'
    params_dip.iterations = 1000

# tv_reconstructor = TVReconstructor(ray_trafo, hyper_params=params_tv.dict)
# dip_reconstructor = DeepImagePriorReconstructor(ray_trafo, hyper_params=params_dip.dict)
reconstructors = [fbp_reconstructor]
options = {'save_iterates': True}
eval_tt.append_all_combinations(reconstructors=reconstructors,
                                test_data=[test_data], options=options)

# run task table
results = eval_tt.run()

running task 0/1 ...


In [5]:
# fbp_reconstructor = FBPReconstructor(ray_trafo, hyper_params={
#                                         'filter_type': 'Hann',
#                                         'frequency_scaling': 0.8})
tv_reconstructor = TVReconstructor(ray_trafo, hyper_params=params_tv.dict)
# dip_reconstructor = DeepImagePriorReconstructor(ray_trafo, hyper_params=params_dip.dict)
reconstructors = [tv_reconstructor]
options = {'save_iterates': True}
eval_tt.append_all_combinations(reconstructors=reconstructors,
                                test_data=[test_data], options=options)

# run task table
results = eval_tt.run()

running task 0/2 ...
running task 1/2 ...


In [6]:
# tv_reconstructor = TVReconstructor(ray_trafo, hyper_params=params_tv.dict)
dip_reconstructor = DeepImagePriorReconstructor(ray_trafo, hyper_params=params_dip.dict)
reconstructors = [dip_reconstructor]
options = {'save_iterates': True}
eval_tt.append_all_combinations(reconstructors=reconstructors,
                                test_data=[test_data], options=options)

# run task table
results = eval_tt.run()
# results.apply_measures([PSNR, SSIM])
# print(results)    3.14s/iteration

running task 0/3 ...
running task 1/3 ...
running task 2/3 ...


100%|██████████| 2000/2000 [10:43<00:00,  3.11it/s]


In [7]:
##############################proposed method ################################
prefix = 'Proposed'
initial_reco = results.results['reconstructions'][0][0][0]
initial_reco = np.asarray(initial_reco)
hyreconstructor = HybirdReconstructor(
    ray_trafo =ray_trafo,
    training_epoch=ITERS,
    layer_num = LAYERS,
    ks        = 64,
    callback_func=hy_callback_func
    )
psnrs, ssims, loss = hyreconstructor.train(observation, initial_reco, ground_truth, prefix, save_path, 'cnn')

This model has 1,036,928 trainable parameters.


HyRecon: 100%|██████████| 2000/2000 [33:43<00:00,  1.01s/it]
